# 10. Собираю обучение и генерацию самостоятельно

**Цель:** связать знакомые части в один работающий процесс и уметь объяснить каждую из них. Это закрепление, не гонка за качеством текста.

Ты уже написал в чате срезы для батча, шаг обучения и цикл генерации. Теперь напишешь их в функциях и соединишь с нашей моделью. Готового решения этих функций здесь нет.

**Готово от ментора:** чтение корпуса, словарь, прежняя архитектура и небольшие проверки для твоего запуска. **Твоя работа:** A — окна и батч; B — loss и обновление; C — цикл обучения; D — генерация.

Начни только с **A** и остановись. На каждую следующую часть переходим после разбора; весь ноутбук не обязательно делать за вечер. Если застрял, пришли код, ожидаемую форму и фактическую ошибку.

Ноутбук — самостоятельный: запускать урок 09 не требуется. Интернет и сохранённые checkpoints не нужны. Ячейки с `NotImplementedError` намеренно ждут твоего решения; удали эту строку после заполнения. Проверки выполняешь ты. Ментор при подготовке обучение не запускал.

## Карта процесса

| Этап | Вход → выход | Что это означает |
|---|---|---|
| Кодирование | текст → `(N,)` | Один ID на символ; здесь токены — символы, не слова |
| Батч | IDs → `xb`, `yb`: `(B,T)` | B последовательностей; target сдвинут на один символ |
| Модель | `(B,T)` → `(B,T,V)` | Для каждой позиции — V оценок следующего токена |
| Loss | `(B*T,V)` и `(B*T,)` → скаляр | Сравниваем каждое предсказание со своим target |
| Обучение | loss → gradients → новые параметры | `backward` и `step` выполняют разные задачи |
| Генерация | последний логит `(B,V)` → ID `(B,1)` | Дописываем по одному символу и повторяем |

`B` — число последовательностей в одном батче, `T` — число токенов в каждой, `V` — размер словаря, `D` — ширина внутреннего вектора. `V` и `D` — не одно и то же.

При обучении ответы берутся из настоящего текста. При генерации следующего настоящего ответа нет: модель использует собственный выбранный токен.

## Подготовка — готовая инфраструктура

Выбери kernel с PyTorch (наша среда `deepschool`). Запускай ячейки сверху вниз. Используем тот же локальный корпус и разбиение: главы I–X — train, XI–XII — validation. Главы XIII–XIV не оцениваем. Источник и ограничения извлечения описаны в `data/captains_daughter/README.md`.

Словарь строим только по train; неизвестный символ кодируем как `<unk>`. Пока не меняй архитектуру и параметры — сначала собери процесс.

In [1]:
import json
import math
import unicodedata
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

SEED = 42
CONTEXT_LENGTH = 32
BATCH_SIZE = 16
TRAIN_STEPS = 300
LEARNING_RATE = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)

candidates = [p for base in [Path.cwd(), *Path.cwd().parents]
              for p in [base, base / 'Seminars' / 'transformers']]
LESSON_DIR = next((p for p in candidates if (p / 'data/captains_daughter/chapters.json').is_file()), None)
if LESSON_DIR is None:
    raise FileNotFoundError('Открой ноутбук внутри проекта deepschool с локальным корпусом урока 09')
chapters = json.loads((LESSON_DIR / 'data/captains_daughter/chapters.json').read_text(encoding='utf-8'))
def normalize(text):
    return unicodedata.normalize('NFC', text).lower()
train_text = normalize('\n\n'.join(c['text'] for c in chapters[:10]))
val_text = normalize('\n\n'.join(c['text'] for c in chapters[10:12]))
tokens = ['<unk>'] + sorted(set(train_text))
stoi = {token: i for i, token in enumerate(tokens)}
vocab_size = len(tokens)
def encode(text):
    return torch.tensor([stoi.get(c, 0) for c in normalize(text)], dtype=torch.long)
def decode(ids):
    return ''.join(tokens[i] for i in ids.tolist())
train_ids, val_ids = encode(train_text), encode(val_text)
print('device:', DEVICE, '| V:', vocab_size)
print('train / val characters:', len(train_ids), len(val_ids))

device: cuda | V: 75
train / val characters: 130505 29122


## A1. Одна пара вход–ответ

На позиции `t` модель учится предсказывать **следующий** токен. Поэтому `y` — такое же по длине окно, но его начало на одну позицию правее.

Срез `data[a:b]` включает `a`, но **не включает** `b`. Чтобы взять T элементов с начала `start`, правая граница должна учитывать и `start`, и T. Для пары окон нужно T+1 исходных токенов.

Напиши `make_example`: верни два одномерных тензора длины `context_length`. Не преобразовывай IDs в float.

<details><summary>Подсказка, если застрял</summary>

Сначала найди конец входного окна. Затем сдвинь обе его границы на 1, чтобы получить target. Не используй размер словаря как границу среза.

</details>

In [2]:
def make_example(data, start, context_length):
    x = data[start:start+context_length]
    y = data[start+1:start+context_length+1]
    return x,y

In [3]:
# Запусти после A1. Числа здесь — учебные IDs, не вход в настоящую модель.
toy = torch.tensor([10, 20, 30, 40, 50, 60, 70])
x, y = make_example(toy, start=2, context_length=3)
assert x.tolist() == [30, 40, 50]
assert y.tolist() == [40, 50, 60]
assert x.shape == y.shape == (3,)
print('A1: окна совпали с ожидаемыми')

A1: окна совпали с ожидаемыми


## A2. Из нескольких окон — один батч

`starts` содержит B начальных позиций. Для каждой получи пару через `make_example`, накопи два списка и собери каждый в тензор через `torch.stack`. Эта операция **создаёт новую ось**: список B тензоров `(T,)` превращается в `(B,T)`.

Функция ниже не выбирает случайные позиции — это отдельно делает готовый помощник. Так можно проверить твои срезы на конкретных числах.

In [4]:
def make_batch(data, starts, context_length):

    xs, ys = list(), list()
    for start in starts:
        x, y = make_example(data, start, context_length)
        xs.append(x)
        ys.append(y)
    xb = torch.stack(xs, dim=0).to(DEVICE)
    yb = torch.stack(ys, dim=0).to(DEVICE)

    return xb, yb

In [5]:
xb, yb = make_batch(toy, starts=[0, 2], context_length=3)
assert xb.tolist() == [[10, 20, 30], [30, 40, 50]]
assert yb.tolist() == [[20, 30, 40], [40, 50, 60]]
assert xb.shape == yb.shape == (2, 3)
assert xb.dtype == yb.dtype == torch.long
print('A2: строки батча и targets верны')

def sample_batch(data, generator):
    # high исключается: последний допустимый start = len(data) - T - 1.
    starts = torch.randint(len(data) - CONTEXT_LENGTH, (BATCH_SIZE,), generator=generator)
    xb, yb = make_batch(data, starts.tolist(), CONTEXT_LENGTH)
    return xb.to(DEVICE), yb.to(DEVICE)

A2: строки батча и targets верны


### Мои ответы — A

1. Почему для одной пары окон длины T нужны T+1 токенов?
2. Что означает строка `xb[1]`: второй батч или вторая последовательность?
3. Какой токен должен предсказываться по последней позиции x в проверке A1?

**Остановка:** пришли «Сделал A». Не нужно заранее решать весь ноутбук.

Ответы:
1. Потому что нам для последнего токена нужен следующий токен для обучения
2. вторая последовательность
3. 60

### Разбор ментора

Исходные ответы и код сохранены. Проверка проведена по коду и сохранённым outputs, без повторного запуска.

Все три ответа A верны. Срезы в make_example и сборка через stack тоже верны: каждая строка — отдельная последовательность, target сдвинут на один токен, последний ответ в примере — 60.

Небольшое замечание к организации: make_batch уже вызывает .to(DEVICE), а sample_batch повторяет это. На том же устройстве это обычно не копирует тензор заново, но ответственность дублируется. Можно оставить перенос только в sample_batch, чтобы make_batch занималась исключительно сборкой окон. Это не ошибка твоих результатов.

## Готовая модель из урока 09

Эти три класса перенесены без изменения вычислений. Здесь нет скрытого импорта или запуска старого ноутбука. Не переписывай attention сейчас.

Внутри: token embedding + position embedding → causal Transformer-блоки → LayerNorm → линейная проекция в словарь. Маска запрещает заглядывать в будущие токены. Поэтому мы можем вычислять loss по всем позициям сразу, хотя генерируем текст по одному токену.

In [6]:
class VectorizedCausalAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model <= 0 or num_heads <= 0 or d_model % num_heads != 0:
            raise ValueError('Нужны положительные D,H и D, кратное H')
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query_projection = nn.Linear(d_model, d_model, bias=False)
        self.key_projection = nn.Linear(d_model, d_model, bias=False)
        self.value_projection = nn.Linear(d_model, d_model, bias=False)
        self.output_projection = nn.Linear(d_model, d_model, bias=False)

    def project_heads(self, x):
        if x.ndim != 3 or x.shape[-1] != self.d_model or x.shape[1] == 0:
            raise ValueError('Ожидается непустой x формы (B,T,D)')
        batch_size, length, _ = x.shape
        # Готовая реализация из урока 05c: проекция и разделение голов.
        q, k, v = [
            projection(x)
            .reshape(batch_size, length, self.num_heads, self.head_dim)
            .transpose(1, 2)
            for projection in (
                self.query_projection,
                self.key_projection,
                self.value_projection,
            )
        ]

        return q, k, v

    def forward(self, x):
        batch_size, length, _ = x.shape
        q, k, v = self.project_heads(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
        positions = torch.arange(x.shape[1], device=x.device)
        blocked = positions[None, :] > positions[:, None]
        masked_scores = scores.masked_fill(blocked, float('-inf'))
        weights = torch.softmax(masked_scores, dim=-1)
        context = weights @ v
        combined = context.transpose(1, 2).reshape(batch_size, length, self.d_model)
        return self.output_projection(combined)

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.norm_1 = nn.LayerNorm(d_model)
        self.attention = VectorizedCausalAttention(d_model, num_heads)
        self.norm_2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        x = x + self.attention(self.norm_1(x))
        x = x + self.mlp(self.norm_2(x))
        return x

In [8]:
class TinyLanguageModel(nn.Module):
    def __init__(self, vocab_size, max_seq_len=16, d_model=8,
                 num_heads=2, d_ff=32, n_layers=1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, token_ids):
        if token_ids.ndim != 2:
            raise ValueError('Ожидаются token IDs формы (B,T)')
        length = token_ids.shape[1]
        if not 1 <= length <= self.max_seq_len:
            raise ValueError('Длина должна быть от 1 до max_seq_len')
        positions = torch.arange(length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits

In [9]:
def new_model():
    return TinyLanguageModel(vocab_size=vocab_size, max_seq_len=CONTEXT_LENGTH,
                             d_model=32, num_heads=4, d_ff=128, n_layers=2).to(DEVICE)

## B1. От logits к одному loss

`logits[b,t]` — V ненормированных оценок следующего токена, а `yb[b,t]` — ID правильного ответа. Нам нужно сравнить B*T таких пар.

Напиши `language_model_loss`: извлеки V из формы logits, объедини B и T через `reshape`, так же разверни targets и передай их в `F.cross_entropy`. **Не добавляй softmax:** эта функция принимает logits. Стандартное усреднение даёт один скаляр по всем B*T позициям.

`reshape` меняет представление формы, не перемешивая порядок пар. `-1` в reshape означает «вычисли размер этой оси». Это другое употребление, чем `dim=-1` — «последняя ось».

In [10]:
def language_model_loss(logits, targets):
    V = logits.shape[-1]
    logits = logits.reshape(-1, V)
    targets = targets.reshape(-1)
    loss = F.cross_entropy(logits, targets)
    return loss

In [11]:
# Одинаковые logits означают равные вероятности: ожидаемый loss = ln(V).
test_logits = torch.zeros(2, 3, 5, requires_grad=True)
test_targets = torch.tensor([[0, 1, 2], [3, 4, 0]])
test_loss = language_model_loss(test_logits, test_targets)
assert test_loss.ndim == 0
assert torch.allclose(test_loss.detach(), torch.tensor(math.log(5)))
print('B1: скалярный loss:', test_loss.item())

B1: скалярный loss: 1.6094379425048828


## B2. Один шаг обучения

Вспомним обязанности:

- `model.train()` выбирает режим обучения; сам по себе не вычисляет градиенты.
- `optimizer.zero_grad(set_to_none=True)` убирает градиенты прошлого шага. Иначе `backward()` будет их накапливать.
- Forward вычисляет значения и при включённом autograd записывает граф операций.
- `loss.backward()` вычисляет производные этого loss по параметрам и накапливает их в `.grad`. Веса ещё не меняются.
- `optimizer.step()` использует `.grad` и обновляет параметры по правилу оптимизатора.
- `loss.item()` возвращает число для журнала, а не тензор с графом. Вызывай backward у loss, не у этого числа.

Напиши шаг сам. Используй свою функцию loss из B1. Возвращай число loss; генерация сюда не входит.

In [12]:
def train_step(model, optimizer, xb, yb):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    logits = model(xb)

    loss = language_model_loss(logits, yb)
    loss.backward()
    optimizer.step()
    return loss.item()

In [13]:
# Проверяем на отдельной модели, чтобы не менять старт будущего обучения.
torch.manual_seed(SEED)
probe = new_model()
probe_optimizer = torch.optim.Adam(probe.parameters(), lr=LEARNING_RATE)
xb, yb = sample_batch(train_ids, torch.Generator().manual_seed(10))
before = probe.lm_head.weight.detach().clone()
value = train_step(probe, probe_optimizer, xb, yb)
assert isinstance(value, float) and math.isfinite(value)
assert probe.lm_head.weight.grad is not None
assert not torch.equal(before, probe.lm_head.weight.detach())
print('B2: loss конечный, градиент есть, веса изменились')
del probe, probe_optimizer

B2: loss конечный, градиент есть, веса изменились


### Мои ответы — B

1. Что изменится после backward без step? А после step?
2. Почему нельзя вызвать backward у `loss.item()`?
3. Почему при logits `(3,5,75)` targets не должны иметь форму `(3,5,75)`?

**Остановка:** разбираем B, прежде чем запускать обучение.

Ответы:
1. Посчитаются градиенты и сохранятся в .grad, после step обновятся веса
2. Потому что это скаляр. Это не граф вычислений
3. потому что у нас в targets хранится только правильный следующий токен, а не логиты на весь словарь

### Разбор ментора

Исходные ответы и код сохранены.

1. Верно: backward вычисляет и накапливает градиенты в .grad, а step обновляет параметры. В твоём шаге перед backward градиенты очищаются, поэтому между шагами они не накапливаются.
2. Важно уточнить: проблема НЕ в скалярности. Сам loss — тоже скаляр, но это тензор формы () со связью с графом. У него можно вызвать backward. loss.item() возвращает обычный Python float: у него нет ни связи с графом, ни метода backward. Разница — Tensor и Python-число, а не скаляр и вектор.
3. Верно для нашей постановки: targets содержит один целочисленный ID на позицию. До reshape форма (3,5), после — (15,). Logits превращаются в (15,75). Cross-entropy также поддерживает вероятностные targets, но это другой режим; мы его здесь не используем.

Код language_model_loss верен. Порядок zero_grad → forward → loss → backward → step в train_step тоже верен. Пропущен model.train() в начале функции. В текущем запуске новая модель уже в режиме train, а evaluate восстанавливает режим, поэтому это не испортило результат. Но generate_ids оставляет модель в eval: перед дальнейшим обучением нужно явно вернуть train. В нашей архитектуре нет dropout и BatchNorm, поэтому сейчас численного отличия режимов нет; с такими слоями оно появится.

Твоя маленькая правка: добавь model.train() в начало train_step самостоятельно.

## C. Соединяем части: короткое обучение

Один шаг — один случайный батч и одно обновление. 300 шагов здесь **не означают 300 эпох**: мы выбираем окна случайно, а не обходим корпус ровно один раз.

Validation ниже готова, чтобы не добавлять ещё одну большую задачу. Она не обновляет параметры; каждый вызов оценивает одни и те же случайно выбранные окна. Это оценка на небольшой выборке окон, не точный loss всей книги. Равные B и T позволяют усреднить batch loss обычным средним.

Training loss из очередного шага шумит и измерен до его обновления. Не сравнивай его с validation как два идеально одинаковых измерения; для начала и конца используем одну и ту же функцию оценки.

In [14]:
@torch.no_grad()
def evaluate(model, data, seed):
    was_training = model.training
    model.eval()
    rng = torch.Generator().manual_seed(seed)
    values = []
    try:
        for _ in range(4):
            xb, yb = sample_batch(data, rng)
            values.append(language_model_loss(model(xb), yb).item())
    finally:
        model.train(was_training)
    return sum(values) / len(values)

# При повторном запуске этой ячейки обучение начинается с нуля.
torch.manual_seed(SEED)
model = new_model()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
train_rng = torch.Generator().manual_seed(43)
history = []
initial_metrics = {'train': evaluate(model, train_ids, 1001),
                   'val': evaluate(model, val_ids, 1002)}
print('До обучения:', initial_metrics)

До обучения: {'train': 4.501575827598572, 'val': 4.5059205293655396}


In [15]:
history_loss = []
for step_id in range(TRAIN_STEPS):
    xb, yb = sample_batch(train_ids, train_rng)
    train_loss = train_step(model, optimizer, xb, yb)
    history_loss.append(train_loss)
    if step_id % 50 == 0:
        print(step_id, ":", train_loss)


0 : 4.507498264312744
50 : 3.2691895961761475
100 : 2.9986817836761475
150 : 2.863313674926758
200 : 2.6894278526306152
250 : 2.7145609855651855


In [16]:
assert len(history_loss) == TRAIN_STEPS, 'Проверь число шагов; перед повтором сбрось модель и history'
assert all(math.isfinite(v) for v in history_loss)
final_metrics = {'train': evaluate(model, train_ids, 1001),
                 'val': evaluate(model, val_ids, 1002)}
print('До:', initial_metrics)
print('После:', final_metrics)
# Убывание каждого отдельного шага не является обязательным условием.

До: {'train': 4.501575827598572, 'val': 4.5059205293655396}
После: {'train': 2.656249701976776, 'val': 2.635077178478241}


### Мои ответы — C

1. Запиши train/val до и после. Что можно заключить, а чего пока нельзя?
2. Почему нельзя пересоздавать модель или Adam на каждой итерации?
3. Если loss одного батча вырос, доказывает ли это ошибку?

**Остановка:** сначала проверим цикл. Не увеличивай число шагов ради красивого текста.

Ответы:
1. До: {'train': 4.501575827598572, 'val': 4.5059205293655396}
После: {'train': 2.656249701976776, 'val': 2.635077178478241}
Loss уменьшился, обучение есть, и на валидации тоже, переобучения нет
2. Потому что он сбросит свои параметры? В нем же есть обучаемые параметры
3. Мне кажется нужно смотреть на нескольких батчах

### Разбор ментора

Исходные ответы и код сохранены.

1. В сохранённом выводе train loss снизился с 4.5016 до 2.6562, validation — с 4.5059 до 2.6351. Это свидетельство улучшения предсказания следующего символа на выбранных train и validation окнах. Формулировка «переобучения нет» слишком сильная: по двум замерам на четырёх батчах мы не доказываем его отсутствие. Точнее: «в этих замерах нет явного признака ухудшения validation при улучшении train». Нужна динамика на нескольких этапах; связность текста этими числами не доказана. Чуть меньший val loss сам по себе нормален: окна разные и могут различаться по сложности.
2. Разделяем два случая. Новая модель заново инициализирует обучаемые веса — накопленное обучение теряется. Новый Adam с теми же параметрами НЕ сбрасывает веса модели, но теряет собственное состояние: оценки первого и второго моментов градиентов и счётчик шагов. Это служебная память оптимизатора, не дополнительные обучаемые веса сети. Если заменить модель, старый optimizer также продолжит ссылаться на старые параметры, пока не будет создан для новой модели.
3. Да: один более трудный случайный батч может дать больший loss. Смотрим устойчивую тенденцию и оценку на фиксированных окнах, а не требуем убывания каждого шага.

Цикл обучения написан верно. history_loss — допустимое имя, и ты согласованно поменял проверку. Старый history = [] в ячейке инициализации теперь не используется — это только лишняя переменная. Повторный запуск одного цикла продолжит обучение ещё на 300 шагов, но очистит history_loss. Для запуска с нуля сначала выполни ячейку создания model и optimizer.

## D. Генерация твоим кодом

На входе `idx` формы `(B,L)` — уже известный текст. L может расти, но наша таблица позиций допускает не больше `context_length` токенов за один forward.

На каждой итерации:

1. Выбери последние T токенов **каждой** строки. Если текста меньше T, бери весь.
2. Получи logits для этого окна.
3. Оставь последнюю позицию всех последовательностей: нужны оценки **следующего** токена.
4. Сделай softmax по словарю. `dim=-1` обозначает последнюю ось, здесь V.
5. Выбери один ID из распределения каждой строки через `torch.multinomial`.
6. Добавь `(B,1)` к полной истории вдоль оси последовательности через `torch.cat`.

`cat` соединяет по существующей оси, а не создаёт новую, в отличие от stack. После шага полная история имеет длину L+1, **не обязательно T+1**. Обрезаем только вход модели, не накопленный текст.

Используй `model.eval()` и `with torch.no_grad():`. Первое меняет поведение модулей вроде dropout, второе отключает запись графа; eval не заменяет no_grad. Ни backward, ни step в генерации не нужны.

В нашей модели нет dropout, но привычку разделять режим и градиенты сохраняем. Temperature пока не добавляем: закрепляем базовый цикл.

In [19]:
@torch.no_grad()
def generate_ids(model, idx, max_new_tokens, context_length):
    model.eval()
    for _ in range(max_new_tokens):
        logits = model(idx[:,-context_length:])  # B, T, V
        last_logits = logits[:, -1, :]             # B,V
        probs = torch.softmax(last_logits, dim=-1)  # B,V
        next_token = torch.multinomial(probs, num_samples=1) #  B,1
        idx = torch.cat([idx, next_token], dim=1)
    # Напиши сам: окно -> logits -> последний logits -> probabilities
    # -> один ID для каждой строки -> дополнение полной истории.
    # Верни idx после цикла. Здесь не кодируем и не декодируем текст.
    return idx

In [20]:
# Небольшая проверка интерфейса; она не доказывает корректность всех срезов.
prompt_ids = encode('отец мой ').unsqueeze(0).to(DEVICE)
torch.manual_seed(2026)
generated = generate_ids(model, prompt_ids.clone(), max_new_tokens=60,
                         context_length=CONTEXT_LENGTH)
assert generated.shape == (1, prompt_ids.shape[1] + 60)
assert torch.equal(generated[:, :prompt_ids.shape[1]], prompt_ids)
assert generated.dtype == torch.long
assert generated.min() >= 0 and generated.max() < vocab_size
print(decode(generated[0].cpu()))

# Две строки и исходная длина больше окна: частые места ошибок.
long_prompt = encode(train_text[:CONTEXT_LENGTH + 5]).to(DEVICE)
two_prompts = long_prompt.unsqueeze(0).repeat(2, 1)
result = generate_ids(model, two_prompts.clone(), 2, CONTEXT_LENGTH)
assert result.shape == (2, CONTEXT_LENGTH + 7)
assert torch.equal(result[:, :two_prompts.shape[1]], two_prompts)
print('D: длина, префиксы и две последовательности проверены')

отец мой н вроя м оствилорету сис, я валь восвово ч:дчндду, забе у.ас
D: длина, префиксы и две последовательности проверены


### Мои ответы — D

1. При logits `(2,8,75)` какая форма нужна перед softmax? А после multinomial?
2. Начальная длина 9, окно 32, сгенерировано 60 токенов. Какова длина результата? Максимальная длина входа модели?
3. Почему при обучении используем все позиции, а при генерации — последнюю?
4. Почему снижение validation loss не обещает осмысленного продолжения?

Бессмыслица после короткого обучения небольшой символьной модели сама по себе не доказывает ошибку в твоём коде. Сначала проверяем процесс, потом отдельно обсуждаем качество.

Ответы:
1. shape = (16, 75), после multinomial (16,)
2. Длина результата 69 токенов
3. Потому что генерируем по одному токену
4. Не знаю

### Разбор ментора

Исходные ответы и код сохранены.

Код генерации верен: окно обрезается только на входе модели, последний logits выбран для всех строк, softmax идёт по словарю, multinomial выбирает по одному ID, cat дописывает полную историю. @torch.no_grad() — корректная замена with torch.no_grad() для всей функции. В сохранённом выводе проверки длины, префикса и двух последовательностей прошли; это не новый запуск ментора.

1. Здесь ошибка в ответе, но не в вычислениях кода. При logits (2,8,75) выражение logits[:, -1, :] даёт (2,75). Softmax сохраняет (2,75). torch.multinomial(probs, num_samples=1) возвращает (2,1), не (16,). Мы выбираем следующий токен для каждой из ДВУХ последовательностей. Форма (16,75) потребовалась бы для loss по ВСЕМ 2*8 позициям при обучении. В комментариях твоей generate_ids у last_logits и probs исправь B,T на B,V; у next_token — B,1.
2. 69 токенов — верно. Максимальная длина одного входа модели — 32. Полная история растёт, но после заполнения окна модель видит только последние 32 токена.
3. Направление верное, дополним причину. При обучении для каждой позиции известен правильный следующий токен, поэтому используем все позиции как обучающие примеры. При генерации нужно продолжить уже имеющийся текст: именно последняя позиция предсказывает первый ещё не добавленный токен. Предсказания более ранних позиций относятся к продолжениям более коротких префиксов.
4. Validation loss проверяет предсказание следующего символа по настоящему предыдущему тексту. При генерации в контекст попадают собственные предсказания, в том числе неудачные. Модель может улучшить частоты букв и короткие сочетания, ещё не научившись строить связные предложения. Маленькая модель, короткое окно и короткое обучение ограничивают результат; уменьшение средней ошибки не гарантирует осмысленность каждой сгенерированной последовательности.

Закрепление: без запуска напиши формы last_logits, probs и next_token, если исходные logits имеют форму (3,12,75). Объясни, почему 12 не входит в число генерируемых за один шаг токенов. После этого вернись к финальному устному объяснению в последней ячейке.

## Финальное закрепление — без нового кода

Объясни своими словами путь одного примера: исходные символы → два окна → logits → loss → gradients → обновлённые веса. Затем объясни, что меняется при генерации.

Критерий завершения: ты можешь не только запустить ячейки, но и показать соответствие `logits[b,t]` и `yb[b,t]`, объяснить backward/step и рост полной истории при ограниченном окне.

После разбора сохраним твои решения и ответы. Следующий урок — чтение результатов и один управляемый эксперимент, а не увеличение архитектуры вслепую.

### Если запутался

Не удаляй попытку. Покажи форму входа, ожидаемую форму выхода и одну конкретную строку. Подсказки будем давать по одному шагу; не требуется помнить весь PyTorch наизусть.

Повторение: [первое обучение](../../notes/interview-prep/08l-first-training.md), [реальный текст и ограничения результатов](../../notes/interview-prep/08m-real-text-training.md).